In [0]:
%run "../includes/configuration"

#### Filtramos las películas con año de lanzamiento mayores o iguales al 2015

In [0]:
movie_df = spark.read.parquet(f"{silver_folder_path}/movies").filter("year_release_date >= 2015")

In [0]:
genre_df = spark.read.parquet(f"{silver_folder_path}/genres")

In [0]:
movie_genre_df = spark.read.parquet(f"{silver_folder_path}/movie_genres")

#### Join "movie_genres" y "genres"

In [0]:
genres_mov_gen_df = movie_genre_df.join(genre_df,
                                    movie_genre_df.genre_id == genre_df.genre_id,
                                    "inner") \
                                    .select(genre_df.genre_name, movie_genre_df.movie_id)

#### Join "movies" y "genres_mov_gen_df"

In [0]:
results_group_movie_genre = movie_df.join(genres_mov_gen_df,
                                    movie_df.movie_id == genres_mov_gen_df.movie_id,
                                    "inner") \
                                    .select(movie_df.year_release_date, movie_df.budget, 
                                            movie_df.revenue, genres_mov_gen_df.genre_name)

#### 1-Calcular el total de presupuesto y total de ingresos de las peliculas agrupadas por el año de lanzamiento y el genero
#### 2-Hacer un ranking particionado por el año de lanzamiento ordenado por el total del presupuesto y el total de ingresos de forma ascendente

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, asc

In [0]:
df_final = results_group_movie_genre.groupBy("year_release_date", "genre_name") \
        .agg(
            sum("budget").alias("total_budget"),
            sum("revenue").alias("total_revenue")
        ) \
        .withColumn("rank", rank().over(Window.partitionBy("year_release_date").orderBy(asc("total_budget"),asc("total_revenue"))))

In [0]:
df_final.write.mode("overwrite").parquet(f"{gold_folder_path}/results_group_movie_genre")

In [0]:
display(spark.read.parquet(f"{gold_folder_path}/results_group_movie_genre"))